# Agents Intro — AI Lab Instructor Notebook

*LLM & Agentic AI · Expert*



5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple
import re
import numpy as np


def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f"Failed: {name}")
    print(f"OK: {name}")

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# Agents Intro — Student Lab

Offline lab: build a minimal agent loop with tool calling + guardrails.

## 0 — Define tools

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple
import re
import numpy as np


def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f"Failed: {name}")
    print(f"OK: {name}")

rng = np.random.default_rng(0)

**Why this works.** # Agents Intro — Instructor Lab

Offline lab: build a minimal agent loop with tool calling + guardrails.

## 0 — Define tools

## Task 2: 0 — Define tools

## 0 — Define tools

Create three deterministic tools with explicit input/output contracts. The point of this section is to make tool calling concrete before you build any agent loop on top of it.

Each tool should:
- read a small structured argument dictionary
- return a structured dictionary result
- fail deterministically for bad lookups instead of hallucinating an answer

In [ ]:
@dataclass
class Tool:
    name: str
    func: Callable[[Dict[str, Any]], Dict[str, Any]]
    cost: float


def tool_add(args: Dict[str, Any]) -> Dict[str, Any]:
    a = int(args['a'])
    b = int(args['b'])
    return {'result': a + b}


def tool_lower(args: Dict[str, Any]) -> Dict[str, Any]:
    s = str(args['s'])
    return {'result': s.lower()}


def tool_lookup_city_country(args: Dict[str, Any]) -> Dict[str, Any]:
    city = str(args['city']).strip().lower()
    table = {
        'paris': 'france',
        'tokyo': 'japan',
        'delhi': 'india',
        'seattle': 'usa',
    }
    if city not in table:
        return {'error': 'unknown_city'}
    return {'result': table[city]}


TOOLBOX: Dict[str, Tool] = {
    'add': Tool('add', tool_add, cost=1.0),
    'lower': Tool('lower', tool_lower, cost=0.2),
    'lookup_city_country': Tool('lookup_city_country', tool_lookup_city_country, cost=0.5),
}

check('toolbox_has_3', len(TOOLBOX) >= 3)


In [ ]:
# Checks
check('toolbox_has_3', len(TOOLBOX) >= 3)

## Task 3: 1 — A simple policy (mock LLM)

## 1 — A simple policy (mock LLM)

Implement a tiny heuristic policy that maps a user prompt to a tool action. This stands in for the LLM decision step in a real agent system.

The policy should recognize:
- arithmetic requests like `add 2 and 3`
- text normalization requests like `lower: HeLLo`
- lookup requests like `country of Tokyo`

If no valid tool call is needed, return `None`.

In [ ]:
@dataclass
class Action:
    tool_name: str
    args: Dict[str, Any]


def policy(prompt: str) -> Optional[Action]:
    """Very small heuristic policy.

    Returns an Action, or None if no tool should be called.
    """
    p = prompt.strip()

    # arithmetic: "add 2 and 3"
    m = re.search(r"add\s+(-?\d+)\s+(and|to)\s+(-?\d+)", p, flags=re.IGNORECASE)
    if m:
        return Action('add', {'a': int(m.group(1)), 'b': int(m.group(3))})

    # lowercasing: "lower: HeLLo"
    m = re.search(r"lower\s*:\s*(.+)$", p, flags=re.IGNORECASE)
    if m:
        return Action('lower', {'s': m.group(1)})

    # lookup: "country of Tokyo"
    m = re.search(r"country\s+of\s+([A-Za-z ]+)$", p, flags=re.IGNORECASE)
    if m:
        return Action('lookup_city_country', {'city': m.group(1).strip()})

    return None

check('policy_add', policy('add 2 and 3').tool_name == 'add')
check('policy_lower', policy('lower: HeLLo').tool_name == 'lower')


In [ ]:
# Checks
check('policy_add', policy('add 2 and 3').tool_name == 'add')
check('policy_lower', policy('lower: HeLLo').tool_name == 'lower')

## Task 4: 2 — Agent loop with guardrails

## 2 — Agent loop with guardrails

Build the minimal agent executor around the policy and tools.

Your loop should enforce production-style controls:
- a tool allowlist
- a total cost budget
- a max-step limit
- deterministic failure when the policy chooses no valid action or a tool returns an error

Return a structured result containing whether the run succeeded, the final answer, the number of steps, total cost, and a short trace.

In [ ]:
$f

In [ ]:
# Checks
check('agent_add', r.ok and r.answer == '15')

## Task 5: 3 — Evaluation harness

## 3 — Evaluation harness

Evaluate the agent over a small benchmark and summarize the operational metrics, not just correctness.

Track at least:
- success rate
- average number of steps
- average tool cost

This mirrors how real agent systems are judged in production: quality alone is not enough if the workflow is too slow or too expensive.

In [ ]:
TASKS = [
    ('add 2 and 3', '5'),
    ('lower: HeLLo', 'hello'),
    ('country of Tokyo', 'japan'),
    ('country of Paris', 'france'),
]


def evaluate(tasks, *, toolbox, allowlist=None):
    oks = 0
    steps = []
    costs = []
    for t, expected in tasks:
        r = run_agent(t, toolbox=toolbox, allowlist=allowlist, max_steps=3, budget=3.0)
        ok = r.ok and (r.answer == expected)
        oks += int(ok)
        steps.append(r.steps)
        costs.append(r.cost)
    return {
        'n': len(tasks),
        'success_rate': oks / len(tasks),
        'avg_steps': float(np.mean(steps)),
        'avg_cost': float(np.mean(costs)),
    }

metrics = evaluate(TASKS, toolbox=TOOLBOX, allowlist=set(TOOLBOX.keys()))
print(metrics)
check('good_success', metrics['success_rate'] >= 0.75)


In [ ]:
# Checks
check('good_success', metrics['success_rate'] >= 0.75)